# Detección de ciclos de Floyd (liebre y tortuga)

Dos punteros recorren una secuencia definida por $x_{k+1} = f(x_k)$: la tortuga avanza un paso y la liebre dos. Si hay ciclo, acaban encontrándose. Después, moviendo uno desde el inicio y otro desde el encuentro a la misma velocidad, se encuentran en la entrada del ciclo. Cuesta $O(n)$ tiempo y $O(1)$ memoria.

## 1. Ciclo en una lista enlazada

Representar una lista enlazada con un diccionario `nodo → siguiente` y decidir si tiene ciclo, devolviendo dónde empieza.

In [1]:
def inicio_ciclo(siguiente, cabeza):
    tortuga = liebre = cabeza
    while liebre is not None and siguiente.get(liebre) is not None:
        tortuga = siguiente[tortuga]
        liebre = siguiente[siguiente[liebre]]
        if tortuga == liebre:
            tortuga = cabeza
            while tortuga != liebre:
                tortuga, liebre = siguiente[tortuga], siguiente[liebre]
            return tortuga
    return None

con_ciclo = {1: 2, 2: 3, 3: 4, 4: 5, 5: 3}
sin_ciclo = {1: 2, 2: 3, 3: None}
print(inicio_ciclo(con_ciclo, 1), inicio_ciclo(sin_ciclo, 1))

3 None


## 2. Número duplicado

Una lista de $n + 1$ enteros con valores entre 1 y $n$ tiene al menos un repetido. Encontrarlo sin modificar la lista y con $O(1)$ memoria, viendo la lista como la función $i \to l[i]$.

In [2]:
def duplicado(lista):
    tortuga = liebre = 0
    while True:
        tortuga = lista[tortuga]
        liebre = lista[lista[liebre]]
        if tortuga == liebre:
            break
    tortuga = 0
    while tortuga != liebre:
        tortuga, liebre = lista[tortuga], lista[liebre]
    return tortuga

print(duplicado([1, 3, 4, 2, 2]), duplicado([3, 1, 3, 4, 2]))

2 3


## 3. Ciclo de una secuencia numérica

La secuencia $x_{k+1} = x_k^2 \bmod 1000$ con $x_0 = 7$ acaba repitiéndose. Calcular la longitud de la parte inicial (antes del ciclo) y la longitud del ciclo.

In [3]:
def floyd(f, x0):
    tortuga, liebre = f(x0), f(f(x0))
    while tortuga != liebre:
        tortuga, liebre = f(tortuga), f(f(liebre))
    mu, tortuga = 0, x0
    while tortuga != liebre:
        tortuga, liebre, mu = f(tortuga), f(liebre), mu + 1
    lam, liebre = 1, f(tortuga)
    while tortuga != liebre:
        liebre, lam = f(liebre), lam + 1
    return mu, lam

print(floyd(lambda x: x * x % 1000, 7))

(2, 4)


## 4. Traza y comprobación del duplicado

Mostrar el recorrido $0 \to l[0] \to l[l[0]] \to \dots$ del ejercicio 2, que tiene forma de $\rho$: una cola y un ciclo cuya entrada es el duplicado. Comprobar `duplicado` contra un recuento con fuerza bruta.

In [4]:
import random
from collections import Counter

def recorrido(lista):
    vistos, actual, camino = set(), 0, []
    while actual not in vistos:
        vistos.add(actual)
        camino.append(actual)
        actual = lista[actual]
    return camino, actual

camino, entrada = recorrido([3, 1, 3, 4, 2])
print(" → ".join(map(str, camino + [entrada])), f"(entrada del ciclo: {entrada})")

random.seed(4)
for _ in range(1000):
    n = random.randint(1, 30)
    datos = list(range(1, n + 1)) + [random.randint(1, n)]
    random.shuffle(datos)
    repetido = next(x for x, c in Counter(datos).items() if c > 1)
    assert duplicado(datos) == repetido == recorrido(datos)[1]
print("Comprobadas 1000 listas")

0 → 3 → 4 → 2 → 3 (entrada del ciclo: 3)
Comprobadas 1000 listas


## 5. Factorizar con rho de Pollard

La secuencia $x_{k+1} = (x_k^2 + c) \bmod n$ se comporta como pseudoaleatoria y, módulo un factor primo $p$ de $n$, entra en un ciclo en unos $\sqrt{p}$ pasos. Floyd lo detecta cuando $\gcd(|x_{\text{tortuga}} - x_{\text{liebre}}|, n) > 1$. Usarlo para factorizar números compuestos y comprobarlo con división por tentativa.

In [5]:
from math import gcd, isqrt

def pollard_rho(n, c=1):
    if n % 2 == 0:
        return 2
    f = lambda x: (x * x + c) % n
    tortuga = liebre = 2
    d = 1
    while d == 1:
        tortuga, liebre = f(tortuga), f(f(liebre))
        d = gcd(abs(tortuga - liebre), n)
    # d == n: el ciclo no separó factores, probar otra constante
    return d if d != n else pollard_rho(n, c + 1)

def es_primo(n):
    return n > 1 and all(n % d for d in range(2, isqrt(n) + 1))

def factorizar(n):
    if n == 1:
        return []
    if es_primo(n):
        return [n]
    d = pollard_rho(n)
    return sorted(factorizar(d) + factorizar(n // d))

def factorizar_bruto(n):
    factores, d = [], 2
    while d * d <= n:
        while n % d == 0:
            factores.append(d)
            n //= d
        d += 1
    return factores + ([n] if n > 1 else [])

for n in range(1, 5000):
    assert factorizar(n) == factorizar_bruto(n)
print(factorizar(8051), factorizar(10403), factorizar(600851475143))

[83, 97] [101, 103] [71, 839, 1471, 6857]
